# 4b. AHP Susceptibility Raster (Optional: Generate AHP Map Using Python)

Generate the AHP-based flood susceptibility map by applying weighted overlay
of 8 reclassified conditioning factor rasters, then classifying the continuous
Flood Susceptibility Index (FSI) into 5 classes using Jenks natural breaks.

## 1. Imports / 

In [ ]:
import numpy as np
import rasterio
import jenkspy
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

os.makedirs('outputs', exist_ok=True)

## 2. Configuration / 

AHP weights derived from pairwise comparison matrix (Notebook 4).
Factor order: elevation, slope, distance_to_river, distance_to_coast, land_cover,
soil_type, ndvi, rainfall.

In [ ]:
# AHP weights (must sum to ~1.0)
# Order: elevation, slope, distance_to_river, distance_to_coast,
# land_cover, soil_type, ndvi, rainfall
WEIGHTS = np.array([0.295, 0.192, 0.133, 0.120,
 0.102, 0.069, 0.056, 0.035])

print(f'Weights sum: {WEIGHTS.sum():.4f}')
print(f'Number of factors: {len(WEIGHTS)}')

## 3. Factor Raster Paths / 

All rasters must share the same CRS (EPSG:32647, UTM Zone 47N) and
spatial resolution (8 m). Place raw GeoTIFFs in `data/raw/`.

In [ ]:
RASTER_DIR = 'data/raw'

FACTOR_FILES = {
    'elevation': os.path.join(RASTER_DIR, 'elevation.tif'),
    'slope': os.path.join(RASTER_DIR, 'slope.tif'),
    'distance_to_river': os.path.join(RASTER_DIR, 'distance_to_river.tif'),
    'distance_to_coast': os.path.join(RASTER_DIR, 'distance_to_coast.tif'),
    'land_cover': os.path.join(RASTER_DIR, 'land_cover.tif'),
    'soil_type': os.path.join(RASTER_DIR, 'soil_type.tif'),
    'ndvi': os.path.join(RASTER_DIR, 'ndvi.tif'),
    'rainfall': os.path.join(RASTER_DIR, 'rainfall.tif'),
}

# Factor order must match WEIGHTS array
FACTOR_NAMES = list(FACTOR_FILES.keys())

for name, path in FACTOR_FILES.items():
    exists = '✓' if os.path.exists(path) else '✗'
    print(f' {exists} {name}: {path}')

## 4. Reclassification Breakpoints / 

Each continuous factor is classified into 5 susceptibility classes (1-5).
- **Elevation & slope**: lower values → more susceptible (inverted scale)
- **Distance factors**: closer → more susceptible
- **NDVI**: lower vegetation index → more susceptible
- **Rainfall**: higher → more susceptible
- **Categorical factors** (land cover, soil type) use direct class mapping

In [ ]:
# Continuous factor reclassification breakpoints
# direction='ascending': higher raw value → higher susceptibility class
# direction='descending': lower raw value → higher susceptibility class
RECLASS_CONTINUOUS = {
    'elevation': {'breaks': [25, 50, 100, 200],
    'direction': 'descending'},
    'slope': {'breaks': [2, 8, 15, 25],
    'direction': 'descending'},
    'distance_to_river': {'breaks': [250, 500, 1000, 2000],
    'direction': 'ascending'},
    'distance_to_coast': {'breaks': [500, 1000, 2000, 5000],
    'direction': 'ascending'},
    'ndvi': {'breaks': [-0.2, 0.1, 0.3, 0.5],
    'direction': 'ascending'},
    'rainfall': {'breaks': [3200, 3500, 3800, 4100],
    'direction': 'ascending'},
}

# Categorical factor reclassification
# Land cover: 1=settlement, 2=agriculture, 3=wetland, 4=forest
# Soil type: 1=Dystric Fluvisols, 2=Humic Acrisols
RECLASS_CATEGORICAL = {
    'land_cover': {1: 5, 2: 3, 3: 4, 4: 1},
    'soil_type': {1: 4, 2: 2},
}

print('Continuous reclassification:')
for name, cfg in RECLASS_CONTINUOUS.items():
    print(f' {name}: breaks={cfg["breaks"]}, {cfg["direction"]}')
print('\nCategorical reclassification:')
for name, mapping in RECLASS_CATEGORICAL.items():
    print(f' {name}: {mapping}')

## 5. Reclassification Functions / 

In [ ]:
from unittest import result


def reclass_continuous(raster_array, breakpoints, direction):
    """Reclassify a continuous raster into 5 classes using fixed breakpoints.

    direction='ascending': higher value → higher class
    direction='descending': lower value → higher class
    """
    result = np.full(raster_array.shape, np.nan, dtype=np.float32)
    valid = np.isfinite(raster_array)
    b0, b1, b2, b3 = breakpoints

    if direction == 'ascending':
        result[valid & (raster_array < b0)] = 1
        result[valid & (raster_array >= b0) & (raster_array < b1)] = 2
        result[valid & (raster_array >= b1) & (raster_array < b2)] = 3
        result[valid & (raster_array >= b2) & (raster_array < b3)] = 4
        result[valid & (raster_array >= b3)] = 5
    else: # descending
        result[valid & (raster_array < b0)] = 5
        result[valid & (raster_array >= b0) & (raster_array < b1)] = 4
        result[valid & (raster_array >= b1) & (raster_array < b2)] = 3
        result[valid & (raster_array >= b2) & (raster_array < b3)] = 2
        result[valid & (raster_array >= b3)] = 1

    return result


def reclass_categorical(raster_array, mapping):
    """Reclassify a categorical raster using a dictionary mapping."""
    result = np.full(raster_array.shape, np.nan, dtype=np.float32)
    valid = np.isfinite(raster_array)
    for old_val, new_val in mapping.items():
        result[valid & (raster_array == old_val)] = new_val
    return result

## 6. Read and Reclassify All Rasters / 

In [ ]:
reclassified = {} # factor_name → 2D array of classes (1-5)
profile = None # rasterio profile for output

for name in FACTOR_NAMES:
    path = FACTOR_FILES[name]
    with rasterio.open(path) as src:
        raw = src.read(1).astype(np.float32)
        nodata_val = src.nodata
    if profile is None:
        profile = src.profile.copy()
        print(f'Reference raster: {name}')
        print(f' CRS: {src.crs}')
        print(f' Shape: {src.width}x{src.height}')
        print(f' Resolution: {src.res}')
        print(f' NoData: {nodata_val}')

    # Replace nodata with NaN for safe processing
    if nodata_val is not None:
        raw[raw == nodata_val] = np.nan

    if name in RECLASS_CONTINUOUS:
        cfg = RECLASS_CONTINUOUS[name]
        rc = reclass_continuous(raw, cfg['breaks'], cfg['direction'])
    else:
        rc = reclass_categorical(raw, RECLASS_CATEGORICAL[name])

    reclassified[name] = rc
    valid_count = np.sum(np.isfinite(rc))
    classes, counts = np.unique(rc[np.isfinite(rc)], return_counts=True)
    print(f'\n{name}: {valid_count:,} valid pixels')
    for c, n in zip(classes, counts):
        print(f' Class {int(c)}: {n:,} pixels ({n/valid_count*100:.1f}%)')

## 7. AHP Weighted Overlay / AHP

$$\text{FSI} = \sum_{i=1}^{8} w_i \times c_i$$

where $w_i$ is the AHP weight and $c_i$ is the reclassified value (1-5) of factor $i$.

In [ ]:
# Stack all reclassified arrays into a 3D cube: (n_factors, rows, cols)
stack = np.stack([reclassified[name] for name in FACTOR_NAMES], axis=0)

# Identify pixels where ALL factors have valid classes
all_valid = np.all(np.isfinite(stack), axis=0)
print(f'Pixels with all 8 factors valid: {all_valid.sum():,}')
print(f'Total pixels: {all_valid.size:,}')
print(f'Coverage: {all_valid.sum() / all_valid.size * 100:.1f}%')

# Compute FSI: weighted sum of classes
# stack shape: (8, rows, cols), weights shape: (8,)
fsi = np.full(stack.shape[1:], np.nan, dtype=np.float32)
fsi[all_valid] = np.sum(stack[:, all_valid] * WEIGHTS[:, np.newaxis], axis=0)

print(f'\nFSI statistics:')
print(f' Min: {np.nanmin(fsi):.3f}')
print(f' Max: {np.nanmax(fsi):.3f}')
print(f' Mean: {np.nanmean(fsi):.3f}')
print(f' Std: {np.nanstd(fsi):.3f}')

## 8. Natural Breaks Classification / 

Classify the continuous FSI into 5 susceptibility classes using
Jenks natural breaks algorithm.

| Class | Label (ID) | Label (EN) |
|-------|------------|------------|
| 1 | Very Low | Very Low |
| 2 | Low | Low |
| 3 | Moderate | Moderate |
| 4 | High | High |
| 5 | Very High | Very High |

In [ ]:
# Extract valid FSI values for natural breaks computation
fsi_valid = fsi[all_valid].astype(np.float64)

# Compute Jenks natural breaks with 5 classes (4 break values)
breaks = jenkspy.jenks_breaks(fsi_valid.tolist(), n_classes=5)
print('Jenks natural breaks:')
for i in range(len(breaks) - 1):
    print(f' Class {i+1}: [{breaks[i]:.3f}, {breaks[i+1]:.3f})')

# Classify FSI into 5 classes
fsi_classified = np.full(fsi.shape, 0, dtype=np.uint8)
fsi_classified[all_valid] = 1 # default to class 1

for i in range(len(breaks) - 1):
    lower = breaks[i]
    upper = breaks[i + 1]
    if i == len(breaks) - 2: # last class includes upper bound
        mask = all_valid & (fsi >= lower) & (fsi <= upper)
    else:
        mask = all_valid & (fsi >= lower) & (fsi < upper)
        fsi_classified[mask] = i + 1

# Verify all valid pixels are classified
classified_count = np.sum(fsi_classified[all_valid] > 0)
print(f'\nClassified pixels: {classified_count:,} / {all_valid.sum():,}')

## 9. Class Distribution Statistics / 

In [ ]:
LABELS = ['Very Low', 'Low', 'Moderate', 'High', 'Very High']
LABELS_EN = ['Very Low', 'Low', 'Moderate', 'High', 'Very High']

total_valid = all_valid.sum()
print('Flood Susceptibility Class Distribution:')
print(f'{"Class":<5} {"Label (ID)":<20} {"Pixels":>10} {"Area (%)":>10}')
print('-' * 50)

dist_rows = []
for i, (label, label_en) in enumerate(zip(LABELS, LABELS_EN), 1):
    count = (fsi_classified[all_valid] == i).sum()
    pct = count / total_valid * 100
    print(f'{i:<5} {label:<20} {count:>10,} {pct:>9.1f}%')
    dist_rows.append({
        'Class': i,
        'Label (ID)': label,
        'Label (EN)': label_en,
        'Pixels': count,
        'Area (%)': round(pct, 2),
        'Break Lower': round(breaks[i-1], 4),
        'Break Upper': round(breaks[i], 4)
    })
print('-' * 50)
print(f'{"Total":<5} {"":20} {total_valid:>10,} {"100.0%":>10}')

dist_df = pd.DataFrame(dist_rows)
print('\nHigh + Very High (susceptible) area:',
    f'{dist_df.loc[dist_df.Class.isin([4,5]), "Area (%)"].sum():.1f}%')
print('Moderate area:',
    f'{dist_df.loc[dist_df.Class == 3, "Area (%)"].values[0]:.1f}%')
print('Low + Very Low area:',
    f'{dist_df.loc[dist_df.Class.isin([1,2]), "Area (%)"].sum():.1f}%')

## 10. Save Susceptibility Raster / 

Output: `outputs/ahp_susceptibility.tif` (GeoTIFF, uint8, classes 1-5).
Nodata = 0.

In [ ]:
output_path = 'outputs/ahp_susceptibility.tif'

# Update profile for output
out_profile = profile.copy()
out_profile.update(
    dtype=rasterio.uint8,
    nodata=0,
    count=1,
    compress='lzw'
)

with rasterio.open(output_path, 'w', **out_profile) as dst:
    dst.write(fsi_classified, 1)
    dst.set_band_description(1, 'AHP Flood Susceptibility (1-5)')

print(f'Saved: {output_path}')
print(f' CRS: {out_profile["crs"]}')
print(f' Shape: {out_profile["width"]}x{out_profile["height"]}')
print(f' dtype: {out_profile["dtype"]}')
print(f' nodata: {out_profile["nodata"]}')

## 11. Save Weights and Statistics / 

In [ ]:
# Save AHP weights
weights_df = pd.DataFrame({
    'Factor': FACTOR_NAMES,
    'Weight': [round(w, 4) for w in WEIGHTS],
    'Percentage': [f'{w*100:.1f}%' for w in WEIGHTS]
})
weights_df.to_csv('outputs/ahp_weights.csv', index=False)
print('AHP weights used for raster overlay:')
print(weights_df.to_string(index=False))

# Save Jenks break values for reproducibility
breaks_df = pd.DataFrame({
    'Class': range(1, 6),
    'Label (ID)': LABELS,
    'Label (EN)': LABELS_EN,
    'Break Lower': [round(breaks[i], 4) for i in range(5)],
    'Break Upper': [round(breaks[i+1], 4) for i in range(5)]
})
breaks_df.to_csv('outputs/ahp_jenks_breaks.csv', index=False)
print('\nJenks natural breaks:')
print(breaks_df.to_string(index=False))

# Save class distribution
dist_df.to_csv('outputs/ahp_class_distribution.csv', index=False)

print('\nSaved:')
print(' outputs/ahp_susceptibility.tif')
print(' outputs/ahp_weights.csv')
print(' outputs/ahp_jenks_breaks.csv')
print(' outputs/ahp_class_distribution.csv')

## 12. Summary / 

**Workflow:**
1. Read 8 factor rasters from `data/raw/`
2. Reclassify each into susceptibility classes (1-5) using fixed breakpoints
3. Apply AHP weighted overlay: FSI = Σ (weight_i × class_i)
4. Classify FSI into 5 classes using Jenks natural breaks
5. Save result as `outputs/ahp_susceptibility.tif`

**Key outputs:**
- `outputs/ahp_susceptibility.tif` — Final AHP susceptibility map (classes 1-5)
- `outputs/ahp_weights.csv` — AHP weights used for overlay
- `outputs/ahp_jenks_breaks.csv` — Jenks break values for reproducibility
- `outputs/ahp_class_distribution.csv` — Area statistics per susceptibility class

**Next steps:**
- The RF-based susceptibility map is generated in Notebook 5 (batch prediction)
- Spatial agreement between AHP and RF maps is analyzed in Notebook 7